# Pipeline de Ingesta: Construcción del Grafo de Conocimiento

Sistema Graph RAG sobre el canon de Sherlock Holmes.
Este notebook ejecuta el pipeline completo de ingesta:
1. Descarga de textos de Project Gutenberg
2. Separación en relatos individuales
3. Chunking consciente de la estructura
4. Extracción multipaso de entidades y relaciones
5. Entity resolution
6. Población del grafo en Neo4j

In [1]:
# Si ejecutas desde notebooks/, necesitas que graphrag sea importable.
# Con uv: uv sync --extra dev && pip install -e .
from graphrag.config import get_settings
from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor
from graphrag.ingestion.entity_extractor import EntityExtractor

settings = get_settings()
print(f"Proyecto GCP: {settings.google_cloud_project}")
print(f"Neo4j URI: {settings.neo4j_uri}")
print(f"Chunk size: {settings.chunk_size}")

Proyecto GCP: holmesgraphrag
Neo4j URI: bolt://localhost:7687
Chunk size: 1500


## 1. Inicializar Neo4j y crear esquema

In [2]:
neo4j = Neo4jManager()
neo4j.setup_database()  # Crea constraints + índices vectoriales + fulltext
print("Base de datos inicializada")
print(f"Stats actuales: {neo4j.get_stats()}")

Base de datos inicializada
Stats actuales: {}


## 2. Descargar y procesar textos de Gutenberg

In [3]:
processor = TextProcessor(neo4j_manager=neo4j)

#Solo los 10 relatos de desarrollo
story_chunks = processor.process_phase1()

print(f"\nRelatos procesados: {len(story_chunks)}")
for title, chunks in story_chunks.items():
    print(f"  - {title}: {len(chunks)} chunks")

Generando embeddings: 100%|██████████| 35/35 [00:00<00:00, 59.08it/s]        
                                                                                            


Relatos procesados: 10
  - A SCANDAL IN BOHEMIA: 32 chunks
  - THE RED-HEADED LEAGUE: 34 chunks
  - A CASE OF IDENTITY: 26 chunks
  - THE FIVE ORANGE PIPS: 27 chunks
  - THE ADVENTURE OF THE BLUE CARBUNCLE: 29 chunks
  - THE ADVENTURE OF THE SPECKLED BAND: 67 chunks
  - THE ADVENTURE OF THE COPPER BEECHES: 37 chunks
  - Silver Blaze: 36 chunks
  - The Final Problem: 27 chunks
  - THE ADVENTURE OF THE DANCING MEN: 35 chunks


## 3. Extracción de entidades y relaciones

Extracción multipaso con sliding context:
- **Paso 1**: Extracción de entidades (Characters, Locations, Crimes, Objects, Deductions, Scenes, Events)
- **Paso 2**: Extracción de relaciones entre las entidades encontradas
- **Entity Resolution**: Normalización + embeddings + LLM para desambiguar duplicados

In [4]:
'''
import logging
logging.getLogger('graphrag.ingestion.entity_extractor').setLevel(logging.DEBUG)

extractor = EntityExtractor()

all_results = {}
story_title = "A SCANDAL IN BOHEMIA"
chunks = story_chunks[story_title]

# Solo 3 chunks para debug rápido
result = extractor.process_story_chunks(chunks, story_title)
all_results[story_title] = result

entities = result["entities"]
print(f"Personajes: {len(entities.get('characters', []))}")
for c in entities["characters"]:
  print(f"  [{c['name']}] aliases: {c.get('aliases', [])}")
'''

'\nimport logging\nlogging.getLogger(\'graphrag.ingestion.entity_extractor\').setLevel(logging.DEBUG)\n\nextractor = EntityExtractor()\n\nall_results = {}\nstory_title = "A SCANDAL IN BOHEMIA"\nchunks = story_chunks[story_title]\n\n# Solo 3 chunks para debug rápido\nresult = extractor.process_story_chunks(chunks, story_title)\nall_results[story_title] = result\n\nentities = result["entities"]\nprint(f"Personajes: {len(entities.get(\'characters\', []))}")\nfor c in entities["characters"]:\n  print(f"  [{c[\'name\']}] aliases: {c.get(\'aliases\', [])}")\n'

In [5]:
extractor = EntityExtractor()

all_results = {}
for story_title, chunks in story_chunks.items():
    print(f"\n{'='*60}")
    print(f"Procesando: {story_title}")
    print(f"{'='*60}")

    result = extractor.process_story_chunks(chunks, story_title)
    all_results[story_title] = result

    # Resumen
    entities = result["entities"]
    n_chars = len(entities.get("characters", []))
    n_locs = len(entities.get("locations", []))
    n_crimes = len(entities.get("crimes", []))
    n_deductions = len(entities.get("deductions", []))
    print(f"  Personajes: {n_chars}, Ubicaciones: {n_locs}, Crímenes: {n_crimes}, Deducciones: {n_deductions}")
    print(f"  Relaciones: {len(result['relationships'])}")


Procesando: A SCANDAL IN BOHEMIA


Extrayendo 'A SCANDAL IN BOHEMIA':  69%|██████▉   | 22/32 [16:46<06:59, 41.93s/it]model_validate_json falló, aplicando extract_json como fallback.
Error actualizando contexto en chunk 23/32: No se pudo extraer JSON de la respuesta
Generando embeddings: 100%|██████████| 87/87 [00:03<00:00, 24.78it/s]


  Personajes: 37, Ubicaciones: 31, Crímenes: 22, Deducciones: 24
  Relaciones: 605

Procesando: THE RED-HEADED LEAGUE


Extrayendo 'THE RED-HEADED LEAGUE':   9%|▉         | 3/34 [02:00<20:02, 38.79s/it]Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 5s.
Error transitorio (intento 2/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 10s.
Extrayendo 'THE RED-HEADED LEAGUE':  85%|████████▌ | 29/34 [18:54<03:28, 41.68s/it]Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 

  Personajes: 27, Ubicaciones: 22, Crímenes: 23, Deducciones: 30
  Relaciones: 588

Procesando: A CASE OF IDENTITY


...

  Personajes: 36, Ubicaciones: 36, Crímenes: 50, Deducciones: 40
  Relaciones: 777

Procesando: The Final Problem


Extrayendo 'The Final Problem':  74%|███████▍  | 20/27 [11:24<04:08, 35.51s/it]model_validate_json falló, aplicando extract_json como fallback.
Error actualizando contexto en chunk 21/27: No se pudo extraer JSON de la respuesta
Generando embeddings: 100%|██████████| 95/95 [00:03<00:00, 27.21it/s]


  Personajes: 23, Ubicaciones: 24, Crímenes: 28, Deducciones: 29
  Relaciones: 496

Procesando: THE ADVENTURE OF THE DANCING MEN


Extrayendo 'THE ADVENTURE OF THE DANCING MEN':  43%|████▎     | 15/35 [09:33<13:33, 40.68s/it]Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 5s.
Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}. Reintentando en 5s.
Extrayendo 'THE ADVENTURE OF THE DANCING MEN':  49%|████▊     | 17/35 [12:17<19:04, 63.59s/it]Error transitorio (intento 1/3): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-42

  Personajes: 28, Ubicaciones: 30, Crímenes: 34, Deducciones: 35
  Relaciones: 757


In [6]:
import json
import os

# Guarda los resultados de extracción a disco por si el pipeline se interrumpe.
# Para recargar sin re-extraer: all_results = json.load(open("../output/extraction_results.json"))
os.makedirs("../output", exist_ok=True)
with open("../output/extraction_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"Checkpoint guardado: ../output/extraction_results.json ({len(all_results)} relatos)")

Checkpoint guardado: ../output/extraction_results.json (10 relatos)


## 4. Poblar el grafo en Neo4j

In [7]:
for story_title, result in all_results.items():
    print(f"Almacenando: {story_title}")

    # Almacenar entidades
    neo4j.store_entities(result["entities"], story_title)

    # Almacenar relaciones
    neo4j.store_relationships(result["relationships"])

    # Vincular chunks con las entidades que mencionan
    for chunk_info in result.get("chunk_entities", []):
        if chunk_info["chunk_id"] and chunk_info["entity_names"]:
            neo4j.link_chunk_to_entities(chunk_info["chunk_id"], chunk_info["entity_names"])

print("\nGrafo poblado exitosamente")

Almacenando: A SCANDAL IN BOHEMIA
Almacenando: THE RED-HEADED LEAGUE
Almacenando: A CASE OF IDENTITY
Almacenando: THE FIVE ORANGE PIPS
Almacenando: THE ADVENTURE OF THE BLUE CARBUNCLE
Almacenando: THE ADVENTURE OF THE SPECKLED BAND
Almacenando: THE ADVENTURE OF THE COPPER BEECHES
Almacenando: Silver Blaze
Almacenando: The Final Problem
Almacenando: THE ADVENTURE OF THE DANCING MEN

Grafo poblado exitosamente


## 5. Verificar el grafo

In [8]:
stats = neo4j.get_stats()
print("Estadísticas del grafo:")
for label, count in stats.items():
    print(f"  {label}: {count}")

Estadísticas del grafo:
  Event: 719
  Object: 580
  Chunk: 350
  Deduction: 317
  Crime: 298
  Scene: 286
  Character: 278
  Location: 261
  Story: 10


In [9]:
# Ver personajes más conectados
top_characters = neo4j.execute_query("""
MATCH (c:Character)-[r]-()
RETURN c.name AS name, count(DISTINCT r) AS connections
ORDER BY connections DESC
LIMIT 10
""")

print("\nPersonajes más conectados:")
for char in top_characters:
    print(f"  {char['name']}: {char['connections']} conexiones")


Personajes más conectados:
  Sherlock Holmes: 185 conexiones
  Mr. Sherlock Holmes: 160 conexiones
  Watson: 99 conexiones
  Dr. Watson: 47 conexiones
  Mister Sherlock Holmes: 38 conexiones
  Dr. Grimesby Roylott: 35 conexiones
  Mr. Jabez Wilson: 33 conexiones
  Miss Mary Sutherland: 33 conexiones
  Mrs. Hilton Cubitt: 33 conexiones
  Miss Violet Hunter: 32 conexiones


In [10]:
# Ver relatos y sus entidades
stories = neo4j.execute_query("""
MATCH (s:Story)
OPTIONAL MATCH (c:Character)-[:APPEARS_IN]->(s)
RETURN s.title AS story, s.collection AS collection, count(c) AS characters
ORDER BY characters DESC
""")

print("\nRelatos cargados:")
for s in stories:
    print(f"  {s['story']} ({s['collection']}): {s['characters']} personajes")


Relatos cargados:
  THE ADVENTURE OF THE SPECKLED BAND (The Adventures of Sherlock Holmes): 55 personajes
  A SCANDAL IN BOHEMIA (The Adventures of Sherlock Holmes): 37 personajes
  Silver Blaze (The Memoirs of Sherlock Holmes): 36 personajes
  THE ADVENTURE OF THE COPPER BEECHES (The Adventures of Sherlock Holmes): 33 personajes
  THE ADVENTURE OF THE BLUE CARBUNCLE (The Adventures of Sherlock Holmes): 30 personajes
  THE FIVE ORANGE PIPS (The Adventures of Sherlock Holmes): 29 personajes
  THE ADVENTURE OF THE DANCING MEN (The Return of Sherlock Holmes): 28 personajes
  THE RED-HEADED LEAGUE (The Adventures of Sherlock Holmes): 27 personajes
  The Final Problem (The Memoirs of Sherlock Holmes): 23 personajes
  A CASE OF IDENTITY (The Adventures of Sherlock Holmes): 22 personajes


In [11]:
# Ver cadenas de deducción
deductions = neo4j.execute_query("""
MATCH (d:Deduction)-[:LEADS_TO]->(d2:Deduction)
RETURN d.observation AS from_obs, d2.observation AS to_obs
LIMIT 5
""")

if deductions:
    print("\nCadenas de deducción encontradas:")
    for d in deductions:
        print(f"  {d['from_obs'][:60]}... → {d['to_obs'][:60]}...")
else:
    print("\nNo se encontraron cadenas de deducción (LEADS_TO)")

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `LEADS_TO` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=23, offset=23>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 23, 'line': 2, 'column': 23}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (d:Deduction)-[:LEADS_TO]->(d2:Deduction)\nRETURN d.observation AS from_obs, d2.observation AS to_obs\nLIMIT 5\n'



No se encontraron cadenas de deducción (LEADS_TO)


## 6. Cleanup (opcional)

In [12]:
 # Descomentar para limpiar la base de datos completa
#neo4j.clear_database()
#print("Base de datos limpiada")

neo4j.close()
print("Conexión cerrada")

Conexión cerrada
